# GalMorphAI · Semana 7 · Avance 5
Ensamble homogéneo y heterogéneo para presencia de anillos. Basado en el notebook ejecutado de Semana 6.

**Estado:** implementación preparada, sin entrenamiento real de Semana 7. Ejecute en Colab o Jupyter con los checkpoints originales. No se fabrican resultados. El test histórico ya fue inspeccionado: no es evaluación independiente. AP media CV es la métrica principal; F1 determina el umbral operativo. La CV de selección/búsqueda es exploratoria y optimista.

Se reutilizan literalmente las definiciones de Semana 6 (código, validación, predictores, pipelines, métricas). No se ejecutan sus celdas de entrenamiento ni exportación. Los ocho resultados históricos quedan embebidos; se reentrenan seis baselines tabulares para una comparación contemporánea. CNN/ViT permanecen como referencia histórica, con coste del extractor separado.


In [9]:
# Ejecutar en un entorno preparado; instalación opcional en Colab:
%pip install 'numpy>=1.26,<3' 'pandas>=2.2,<3' 'scikit-learn>=1.7,<2' 'matplotlib>=3.8,<4' 'joblib>=1.4,<2'

In [10]:
"""Modelado binario GalMorphAI; sin ejecución automática al importar."""
import io,json,hashlib,uuid,platform
from pathlib import Path
from decimal import Decimal,InvalidOperation
from datetime import datetime,timezone
from importlib.metadata import version
import numpy as np
import pandas as pd
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
from sklearn.linear_model import LogisticRegression
from sklearn.svm import SVC
from sklearn.tree import DecisionTreeClassifier
from sklearn.neighbors import KNeighborsClassifier
from sklearn.naive_bayes import GaussianNB
from sklearn.neural_network import MLPClassifier
from sklearn.preprocessing import StandardScaler
from sklearn.compose import ColumnTransformer
from sklearn.model_selection import ParameterSampler
from time import perf_counter
import warnings
from sklearn.model_selection import train_test_split,StratifiedKFold
from sklearn.utils.class_weight import compute_sample_weight
from sklearn.metrics import (average_precision_score,roc_auc_score,balanced_accuracy_score,
    precision_score,recall_score,f1_score,confusion_matrix,precision_recall_curve,roc_curve)
from sklearn.inspection import permutation_importance
import joblib

LABEL_MAP={0:'ninguno',2:'nuclear',4:'interno',8:'externo',12:'interno+externo',16:'parcial'}
SHAPE_SUFFIXES=['r20_over_R','r50_over_R','r80_over_R','concentration_fixed',
 'gini_fixed','asymmetry_fixed','axis_ratio_moments','centroid_offset_over_R',
 'inner_fraction','outer_edge_fraction','radial_peak_over_R','radial_peak_contrast']+[
 f'radial_{i:02d}_relative' for i in range(16)]
PREDICTORS=[b+'_'+s for b in 'grz' for s in SHAPE_SUFFIXES]
MODEL_NAMES=['RegresionLogistica','SVM','ArbolDecision','KNN','GaussianNB','MLP']
MODEL_CFG=dict(version='2.1.0',seed=42,test_fraction=.2,folds=5,
               search_iterations=24,n_jobs=2,min_per_class=30,mlp_max_iter=500,
               objetivo='0=sin_anillo;2,4,8,12,16=con_anillo',
               seleccion='AP media CV; desempate exacto por tiempo de inferencia',
               umbral='F1 OOF entrenamiento')
SEARCH_SPACES={
 'RegresionLogistica':{'model__C':[.001,.01,.1,1,10,100], 'model__class_weight':[None,'balanced'], 'model__solver':['lbfgs','liblinear']},
 'SVM':{'model__C':[.01,.1,1,10,100], 'model__gamma':['scale',.001,.01,.1], 'model__kernel':['rbf','sigmoid'], 'model__class_weight':[None,'balanced']},
 'ArbolDecision':{'model__max_depth':[3,5,8,12,None], 'model__min_samples_leaf':[1,3,8,16], 'model__criterion':['gini','entropy'], 'model__class_weight':[None,'balanced']},
 'KNN':{'model__n_neighbors':[3,5,9,15,25,35], 'model__weights':['uniform','distance'], 'model__p':[1,2]},
 'GaussianNB':{'model__var_smoothing':list(np.logspace(-12,-2,31))},
 'MLP':{'model__hidden_layer_sizes':[(32,),(64,),(64,32),(128,64)], 'model__activation':['relu','tanh'], 'model__alpha':[.0001,.001,.01,.1], 'model__learning_rate_init':[.0001,.001,.01]}}
MODEL_NOTES={
 'RegresionLogistica':'Coeficientes interpretables; frontera lineal; escalado necesario.',
 'SVM':'Frontera no lineal; interpretación limitada; coste depende de vectores de soporte.',
 'ArbolDecision':'Reglas inspeccionables; profundidad controla complejidad y sobreajuste.',
 'KNN':'Predicción por vecinos; sensible a escala y dimensión; conserva entrenamiento.',
 'GaussianNB':'Rápido; supone independencia condicional y distribuciones gaussianas.',
 'MLP':'Red neuronal individual; no lineal, poco interpretable y sensible a convergencia.'}


def msha(data):return hashlib.sha256(data).hexdigest()
def mjson(x):return json.dumps(x,ensure_ascii=False,sort_keys=True,allow_nan=False).encode()


def label_code(value):
    try:
        d=Decimal(str(value).strip())
        if not d.is_finite() or d not in [Decimal(x) for x in LABEL_MAP]:raise ValueError('Código desconocido')
        return int(d)
    except InvalidOperation as e:raise ValueError('Etiqueta no numérica') from e


def load_dataset(csv_bytes,report_bytes,expected_rows=8527):
    report=json.loads(report_bytes)
    # Preservar IDs y etiquetas originales como texto; convertir sólo predictores explícitos.
    df=pd.read_csv(io.BytesIO(csv_bytes),dtype=str,keep_default_na=False)
    required={'objeto_id','indice_fila','etiqueta_original','estado_features',
              'estado_preprocesamiento',*PREDICTORS}
    if not required<=set(df.columns):raise ValueError('Faltan columnas de features: '+str(sorted(required-set(df.columns))))
    if len(df)!=expected_rows or report['resumen']['filas']!=expected_rows:
        raise ValueError('Número de filas incompatible con el catálogo')
    if df.indice_fila.duplicated().any() or df.objeto_id.eq('').any():raise ValueError('Índices repetidos o IDs vacíos')
    states=df.estado_features.value_counts().to_dict()
    if states!=report['resumen']['estados']:raise ValueError('CSV y JSON tienen estados distintos; usar el mismo checkpoint')
    if states.get('pendiente_features',0)>0:
        raise ValueError(f"Faltan {states['pendiente_features']} filas por extraer. Terminar notebook 02 con LIMITE=0.")
    allowed={'extraido','revisar_features','error_features','sin_imagen_aprobada'}
    if not set(states)<=allowed:raise ValueError('Estados desconocidos')
    df['anillos_codigo']=df.etiqueta_original.map(label_code)
    df['tipo_anillo']=df.anillos_codigo.map(LABEL_MAP)
    df['target_binario']=(df.anillos_codigo!=0).astype(int)
    for col in PREDICTORS:
        # Blancos son nulos legítimos; texto no numérico e infinito se rechazan.
        values=pd.to_numeric(df[col].replace('',np.nan),errors='raise')
        if np.isinf(values.to_numpy(dtype=float)).any():raise ValueError('Feature infinita: '+col)
        df[col]=values.astype(float)
    # Contrastar valores elegibles con el JSON hermano, no sólo su recuento.
    for row in df.loc[df.estado_features.isin(['extraido','revisar_features'])].to_dict('records'):
        result=report['resultados'].get(row['objeto_id'])
        if not result or result['estado']!=row['estado_features']:raise ValueError('Objeto ausente/incompatible en JSON')
        for col in PREDICTORS:
            v=result['features'].get(col);actual=row[col]
            if v is None:
                if not pd.isna(actual):raise ValueError('CSV/JSON difieren en nulo: '+col)
            elif not np.isclose(actual,float(v),rtol=1e-12,atol=1e-14,equal_nan=False):
                raise ValueError('CSV/JSON difieren en feature: '+col)
    # El baseline excluye casos QC pendientes de revisión; registra todas las exclusiones.
    excluded=df.loc[df.estado_features!='extraido'].copy()
    eligible=df.loc[df.estado_features=='extraido'].copy()
    if not eligible.estado_preprocesamiento.eq('aprobado').all():raise ValueError('Features sin preprocesamiento aprobado')
    if (df.groupby('objeto_id').anillos_codigo.nunique()>1).any():raise ValueError('Etiquetas contradictorias por objeto')
    if (df.groupby('objeto_id').estado_features.nunique()>1).any():raise ValueError('Estados contradictorios por objeto')
    for _,group in eligible.groupby('objeto_id'):
        if len(group)>1 and not group[PREDICTORS].eq(group[PREDICTORS].iloc[0]).fillna(False).all().all():
            # Comparar NaN como iguales, sin aceptar diferencias numéricas.
            arr=group[PREDICTORS].to_numpy()
            if not np.allclose(arr,arr[0],rtol=0,atol=0,equal_nan=True):raise ValueError('Features contradictorias en duplicados')
    eligible=eligible.sort_values('objeto_id').drop_duplicates('objeto_id').reset_index(drop=True)
    coverage=pd.crosstab(df.tipo_anillo,df.estado_features).reindex(LABEL_MAP.values(),fill_value=0)
    summary=dict(filas_total=len(df),filas_excluidas=len(excluded),objetos_elegibles=len(eligible),
        duplicados_colapsados=int((df.estado_features=='extraido').sum()-len(eligible)),
        estados=states,clases_elegibles=eligible.target_binario.value_counts().to_dict(),
        codigos=LABEL_MAP,semantica_origen='Definición proporcionada y aceptada por el usuario, 17-10-2026',
        csv_sha256=msha(csv_bytes),json_sha256=msha(report_bytes))
    return eligible,excluded,coverage,summary


def split_dataset(df,cfg):
    counts=df.target_binario.value_counts()
    if set(counts.index)!={0,1} or counts.min()<cfg['min_per_class']:
        raise ValueError('Muestras insuficientes en alguna clase binaria')
    train,test=train_test_split(np.arange(len(df)),test_size=cfg['test_fraction'],
                               stratify=df.target_binario,random_state=cfg['seed'])
    if set(df.iloc[train].objeto_id)&set(df.iloc[test].objeto_id):raise ValueError('Objeto compartido entre train/test')
    return train,test


def model_features(name,cfg):
    return cfg.get('image_features',{}).get(name,PREDICTORS)


def experiment_features(cfg):
    return list(dict.fromkeys(PREDICTORS+[c for v in cfg.get('image_features',{}).values() for c in v]))


def make_model(name,cfg):
    estimators={
      'RegresionLogistica':lambda:LogisticRegression(max_iter=2000,class_weight='balanced',random_state=cfg['seed']),
      'SVM':lambda:SVC(probability=True,class_weight='balanced',random_state=cfg['seed']),
      'ArbolDecision':lambda:DecisionTreeClassifier(max_depth=8,min_samples_leaf=3,class_weight='balanced',random_state=cfg['seed']),
      'KNN':lambda:KNeighborsClassifier(n_neighbors=9,n_jobs=cfg['n_jobs']),
      'GaussianNB':lambda:GaussianNB(),
      'MLP':lambda:MLPClassifier(hidden_layer_sizes=(64,),max_iter=cfg['mlp_max_iter'],random_state=cfg['seed'])}
    if name in cfg.get('image_features',{}):
        # Cabeza lineal binaria sobre un extractor neuronal congelado: un modelo, no un ensamble.
        estimators[name]=lambda:LogisticRegression(max_iter=2000,class_weight='balanced',random_state=cfg['seed'])
    if name not in estimators:raise ValueError(name)
    return Pipeline([('columns',ColumnTransformer([('selected','passthrough',model_features(name,cfg))],remainder='drop')),('imputer',SimpleImputer(strategy='median',add_indicator=True,keep_empty_features=True)),
        ('scaler',StandardScaler()),('model',estimators[name]())])


def fit_balanced(model,X,y):
    # class_weight='balanced' se estima dentro del fit para los modelos que lo admiten.
    # KNN, NB y MLP conservan la distribución original, sin remuestreo artificial.
    model.fit(X,y)
    return model


def select_threshold(y,prob):
    thresholds=np.unique(np.r_[0.5,np.linspace(.05,.95,91)])
    scores=[f1_score(y,prob>=t,zero_division=0) for t in thresholds]
    # Desempate determinista: preferir umbral más próximo a 0.5.
    best=sorted(range(len(scores)),key=lambda i:(-scores[i],abs(thresholds[i]-.5),thresholds[i]))[0]
    return float(thresholds[best]),float(scores[best])


def metrics(y,prob,threshold):
    pred=(prob>=threshold).astype(int)
    return dict(average_precision=float(average_precision_score(y,prob)),
        roc_auc=float(roc_auc_score(y,prob)),balanced_accuracy=float(balanced_accuracy_score(y,pred)),
        precision=float(precision_score(y,pred,zero_division=0)),recall=float(recall_score(y,pred,zero_division=0)),
        f1=float(f1_score(y,pred,zero_division=0)),confusion_matrix=confusion_matrix(y,pred,labels=[0,1]).tolist(),
        prevalencia_positiva=float(np.mean(y)))


def evaluate_candidate(name,params,X,y,cv,cfg):
    oof=np.full(len(y),np.nan);rows=[]
    for fold,(a,b) in enumerate(cv,1):
        model=make_model(name,cfg).set_params(**params)
        with warnings.catch_warnings(record=True) as caught:
            warnings.simplefilter('always')
            start=perf_counter();fit_balanced(model,X.iloc[a],y[a]);fit_s=perf_counter()-start
            start=perf_counter();prob=model.predict_proba(X.iloc[b])[:,1];predict_s=perf_counter()-start
        oof[b]=prob
        row=metrics(y[b],prob,.5);row.pop('confusion_matrix')
        rows.append(dict(row,fold=fold,fit_seconds=fit_s,predict_seconds=predict_s,
                         warnings=' | '.join(sorted(set(str(w.message) for w in caught)))))
    if not np.isfinite(oof).all():raise ValueError('OOF incompleto')
    table=pd.DataFrame(rows)
    summary={k:float(table[k].mean()) for k in ['average_precision','roc_auc','balanced_accuracy','precision','recall','f1','fit_seconds','predict_seconds']}
    summary['ap_std']=float(table.average_precision.std(ddof=1))
    summary['warnings_count']=int(table.warnings.ne('').sum())
    return summary,oof,table


def train_baselines(df,cfg):
    train_idx,test_idx=split_dataset(df,cfg)
    train=df.iloc[train_idx].copy();test=df.iloc[test_idx].copy()
    X=train[experiment_features(cfg)];y=train.target_binario.to_numpy()
    folds=min(cfg['folds'],int(pd.Series(y).value_counts().min()))
    if folds<3:raise ValueError('Se necesitan al menos 3 folds')
    cv=list(StratifiedKFold(n_splits=folds,shuffle=True,random_state=cfg['seed']).split(X,y))
    base=[];details=[];cache={};probs={}
    for name in MODEL_NAMES+list(cfg.get('image_features',{})):
        stats,oof,table=evaluate_candidate(name,{},X,y,cv,cfg)
        base.append(dict(modelo=name,**stats));cache[name]=(stats,oof,{})
        probs[name]=oof;details.append(table.assign(modelo=name,etapa='inicial',candidato=0))
        print('Inicial:',name,'AP=',round(stats['average_precision'],4),flush=True)
    comparison=pd.DataFrame(base).sort_values(['average_precision','predict_seconds','modelo'],ascending=[False,True,True]).reset_index(drop=True)
    top2=comparison.modelo.head(2).tolist();search=[];finalists=[]
    for name in top2:
        # Incluir configuración inicial: el ajuste no obliga a empeorar ni a inventar mejora.
        best_stats,best_oof,best_params=cache[name]
        search.append(dict(modelo=name,candidato=0,parametros='{}',**best_stats))
        candidates=list(ParameterSampler((SEARCH_SPACES['RegresionLogistica'] if name in cfg.get('image_features',{}) else SEARCH_SPACES[name]),n_iter=cfg['search_iterations'],random_state=cfg['seed']))
        for number,params in enumerate(candidates,1):
            stats,oof,table=evaluate_candidate(name,params,X,y,cv,cfg)
            search.append(dict(modelo=name,candidato=number,parametros=json.dumps(params),**stats))
            details.append(table.assign(modelo=name,etapa='ajuste',candidato=number))
            if stats['average_precision']>best_stats['average_precision']:
                best_stats,best_oof,best_params=stats,oof,params
            print('Ajuste:',name,number,'/',len(candidates),'AP=',round(stats['average_precision'],4),flush=True)
        cache[name]=(best_stats,best_oof,best_params)
        initial=float(comparison.set_index('modelo').loc[name,'average_precision'])
        finalists.append(dict(modelo=name,ap_inicial=initial,mejora_ap=best_stats['average_precision']-initial,
            parametros=json.dumps(best_params),consideraciones=MODEL_NOTES.get(name,'Extractor neuronal congelado; se ajusta sólo la cabeza. Interpretabilidad limitada y coste adicional de extracción.'),**best_stats))
    tuned=pd.DataFrame(finalists).sort_values(['average_precision','predict_seconds','modelo'],ascending=[False,True,True]).reset_index(drop=True)
    winner=tuned.iloc[0].modelo;stats,oof,params=cache[winner]
    threshold,oof_f1=select_threshold(y,oof)
    start=perf_counter();selected=fit_balanced(make_model(winner,cfg).set_params(**params),X,y)
    final_fit_s=perf_counter()-start
    # El test nunca participa en ranking, búsqueda ni umbral.
    start=perf_counter();test_prob=selected.predict_proba(test[experiment_features(cfg)])[:,1];test_predict_s=perf_counter()-start
    test_metrics=metrics(test.target_binario.to_numpy(),test_prob,threshold)
    predictions=test[['objeto_id','etiqueta_original','anillos_codigo','tipo_anillo','target_binario']].copy()
    predictions['score_con_anillo']=test_prob;predictions['prediccion_binaria']=(test_prob>=threshold).astype(int)
    subtype=[]
    for code,name in LABEL_MAP.items():
        sub=predictions[predictions.anillos_codigo==code]
        subtype.append(dict(codigo=code,tipo=name,n_test=len(sub),fraccion_clasificada_con_anillo=float(sub.prediccion_binaria.mean()) if len(sub) else None))
    winner_features=model_features(winner,cfg)
    importance=permutation_importance(selected,X[winner_features],y,scoring='average_precision',n_repeats=3,random_state=cfg['seed'],n_jobs=1)
    importance_table=pd.DataFrame(dict(feature=winner_features,importance_train=importance.importances_mean,std_train=importance.importances_std)).sort_values('importance_train',ascending=False)
    split=df[['objeto_id','etiqueta_original','tipo_anillo']].copy();split['split']='train';split.loc[test.index,'split']='test'
    oof_table=train[['objeto_id','target_binario']].copy()
    for name,p in probs.items():oof_table[name+'_inicial_oof']=p
    oof_table['final_ajustado_oof']=oof
    runner=tuned.iloc[1]
    justification=(f"Se elige {winner} por AP media CV={stats['average_precision']:.4f}, frente a "
        f"{runner.modelo}={runner.average_precision:.4f}. F1 CV a umbral 0.5={stats['f1']:.4f}; "
        f"sensibilidad={stats['recall']:.4f}; ajuste medio/fold={stats['fit_seconds']:.3f}s; "
        f"inferencia media/fold={stats['predict_seconds']:.3f}s. {MODEL_NOTES.get(winner,'Extractor neuronal congelado; clasificación por cabeza lineal. Interpretabilidad limitada; extracción de imágenes tiene coste adicional.')} "
        'Se prioriza el ranking de candidatos con anillo; se acepta esta limitación de interpretación/coste. '
        'Las diferencias de CV son descriptivas, no prueba de significación. Revisar alertas de convergencia antes de entregar.')
    summary=dict(modelo_elegido=winner,top2=top2,parametros_finales=params,justificacion=justification,
        cv_average_precision=comparison.set_index('modelo').average_precision.to_dict(),umbral=threshold,
        f1_oof_para_umbral=oof_f1,metricas_test=test_metrics,por_tipo=subtype,n_train=len(train),n_test=len(test),folds=folds,
        final_fit_seconds=final_fit_s,test_predict_seconds=test_predict_s,
        test_previamente_inspeccionado=cfg.get('test_previamente_inspeccionado',False),
        vision=cfg.get('vision_provenance'),
        nota='CV reutilizada para selección y ajuste: optimista. Test sólo es independiente si no fue inspeccionado en iteraciones anteriores; scores no necesariamente calibrados.',
        sesgos='Split por objeto, no espacial; exclusiones QC y desbalance por subtipo limitan conclusiones.')
    return dict(model=selected,summary=summary,cv=pd.concat(details,ignore_index=True),comparativa=comparison,
        busqueda=pd.DataFrame(search),ajustados=tuned,split=split,test_predictions=predictions,oof=oof_table,importance=importance_table)


def save_training(result,excluded,coverage,input_summary,cfg,output_root):
    folder=Path(output_root)/(datetime.now(timezone.utc).strftime('%Y%m%dT%H%M%S')+'-'+uuid.uuid4().hex[:12])
    folder.mkdir(parents=True,exist_ok=False)
    def save(name,data):
        with (folder/name).open('xb') as f:f.write(data)
    provenance=dict(config=cfg,entrada=input_summary,labels=LABEL_MAP,predictors=model_features(result['summary']['modelo_elegido'],cfg),
        versiones={p:version(p) for p in ['numpy','pandas','scikit-learn','joblib']},
        python=platform.python_version(),fecha_utc=datetime.now(timezone.utc).isoformat())
    save('provenance.json',mjson(provenance));save('metricas.json',mjson(result['summary']))
    for key in ['cv','comparativa','busqueda','ajustados','split','test_predictions','oof','importance']:
        save(key+'.csv',result[key].to_csv(index=False).encode())
    save('excluidos.csv',excluded[['objeto_id','etiqueta_original','estado_features','estado_preprocesamiento']].to_csv(index=False).encode())
    save('cobertura_por_tipo.csv',coverage.to_csv().encode())
    buffer=io.BytesIO();joblib.dump(dict(pipeline=result['model'],threshold=result['summary']['umbral'],
        features=model_features(result['summary']['modelo_elegido'],cfg),labels=LABEL_MAP,model_name=result['summary']['modelo_elegido'],vision=cfg.get('vision_provenance')),buffer);save('modelo.joblib',buffer.getvalue())
    save('checksums.json',mjson({p.name:msha(p.read_bytes()) for p in folder.iterdir() if p.is_file()}))
    return folder


In [11]:
HISTORICO = {'modelo_elegido': 'MLP', 'top2': ['MLP', 'SVM'], 'parametros_finales': {'model__learning_rate_init': 0.0001, 'model__hidden_layer_sizes': [64], 'model__alpha': 0.001, 'model__activation': 'relu'}, 'justificacion': 'Se elige MLP por AP media CV=0.5866, frente a SVM=0.5607. F1 CV a umbral 0.5=0.5118; sensibilidad=0.4312; ajuste medio/fold=15.019s; inferencia media/fold=0.013s. Red neuronal individual; no lineal, poco interpretable y sensible a convergencia. Se prioriza el ranking de candidatos con anillo; se acepta esta limitación de interpretación/coste. Las diferencias de CV son descriptivas, no prueba de significación. Revisar alertas de convergencia antes de entregar.', 'cv_average_precision': {'MLP': 0.5626972166681523, 'SVM': 0.5607040988586689, 'RegresionLogistica': 0.5478388359067671, 'KNN': 0.48786379879170527, 'ViT_B16': 0.46241625296347805, 'CNN_ResNet18': 0.44079657013128315, 'ArbolDecision': 0.4198941018734016, 'GaussianNB': 0.29982141392798806}, 'umbral': 0.30999999999999994, 'f1_oof_para_umbral': 0.5835748792270531, 'metricas_test': {'average_precision': 0.5889975289730689, 'roc_auc': 0.8394041440105453, 'balanced_accuracy': 0.7436459878068875, 'precision': 0.5021834061135371, 'recall': 0.6845238095238095, 'f1': 0.5793450881612091, 'confusion_matrix': [[928, 228], [106, 230]], 'prevalencia_positiva': 0.225201072386059}, 'por_tipo': [{'codigo': 0, 'tipo': 'ninguno', 'n_test': 1156, 'fraccion_clasificada_con_anillo': 0.1972318339100346}, {'codigo': 2, 'tipo': 'nuclear', 'n_test': 24, 'fraccion_clasificada_con_anillo': 0.4166666666666667}, {'codigo': 4, 'tipo': 'interno', 'n_test': 151, 'fraccion_clasificada_con_anillo': 0.6887417218543046}, {'codigo': 8, 'tipo': 'externo', 'n_test': 38, 'fraccion_clasificada_con_anillo': 0.8947368421052632}, {'codigo': 12, 'tipo': 'interno+externo', 'n_test': 69, 'fraccion_clasificada_con_anillo': 0.7971014492753623}, {'codigo': 16, 'tipo': 'parcial', 'n_test': 54, 'fraccion_clasificada_con_anillo': 0.5}], 'n_train': 5968, 'n_test': 1492, 'folds': 5, 'final_fit_seconds': 18.54028931100038, 'test_predict_seconds': 0.012433746000169776, 'test_previamente_inspeccionado': True, 'vision': {'recipe': {'version': '1.0.0', 'bands': 'grz', 'source': 'NORMALIZED', 'size': 224, 'percentiles': [1, 99], 'normalization': 'ImageNet mean/std sobre canales grz (no RGB físico)', 'cnn': 'ResNet18_Weights.IMAGENET1K_V1', 'vit': 'ViT_B_16_Weights.IMAGENET1K_V1', 'cnn_dim': 512, 'vit_dim': 768, 'training': 'backbone congelado; cabeza logística binaria ajustada por CV', 'torch': '2.11.0+cu130', 'torchvision': '0.26.0+cu130', 'weights_sha256': {'CNN_ResNet18': 'f37072fd47e89c5e827621c5baffa7500819f7896bbacec160b1a16c560e07ec', 'ViT_B16': 'c867db91d3e12c6cbadabb610d73c24a546bf82d8c03a9fea34f43a712ddb0e9'}}, 'cache_id': 'fc32b91a06f88ac864fe211c', 'device': 'cuda', 'objects': 7460, 'uncached_objects': 7460, 'backbone_seconds': {'CNN_ResNet18': 17.811179989006234, 'ViT_B16': 112.4489893919951}, 'wall_seconds': 10379.010398892999, 'nota_tiempos': 'Backbones medidos sólo para objetos no cacheados; no comparar con tiempos de cabeza como si fueran end-to-end.'}, 'nota': 'CV reutilizada para selección y ajuste: optimista. Test sólo es independiente si no fue inspeccionado en iteraciones anteriores; scores no necesariamente calibrados.', 'sesgos': 'Split por objeto, no espacial; exclusiones QC y desbalance por subtipo limitan conclusiones.'}
HISTORICO_TABLA = [{'modelo': 'MLP', 'average_precision': '0.562697', 'roc_auc': '0.810924', 'balanced_accuracy': '0.692717', 'precision': '0.574494', 'recall': '0.492937', 'f1': '0.528893', 'fit_seconds': '13.379517', 'predict_seconds': '0.011457', 'ap_std': '0.054918', 'warnings_count': '0'}, {'modelo': 'SVM', 'average_precision': '0.560704', 'roc_auc': '0.825072', 'balanced_accuracy': '0.676158', 'precision': '0.605437', 'recall': '0.434944', 'f1': '0.504704', 'fit_seconds': '7.583515', 'predict_seconds': '0.434863', 'ap_std': '0.047824', 'warnings_count': '5'}, {'modelo': 'RegresionLogistica', 'average_precision': '0.547839', 'roc_auc': '0.810890', 'balanced_accuracy': '0.741669', 'precision': '0.438855', 'recall': '0.769517', 'f1': '0.558860', 'fit_seconds': '0.750886', 'predict_seconds': '0.026128', 'ap_std': '0.053095', 'warnings_count': '0'}, {'modelo': 'KNN', 'average_precision': '0.487864', 'roc_auc': '0.785304', 'balanced_accuracy': '0.641199', 'precision': '0.587217', 'recall': '0.354647', 'f1': '0.441468', 'fit_seconds': '0.091354', 'predict_seconds': '0.147139', 'ap_std': '0.034729', 'warnings_count': '0'}, {'modelo': 'ViT_B16', 'average_precision': '0.462416', 'roc_auc': '0.738480', 'balanced_accuracy': '0.667968', 'precision': '0.399967', 'recall': '0.597026', 'f1': '0.478925', 'fit_seconds': '2.297461', 'predict_seconds': '0.027583', 'ap_std': '0.025392', 'warnings_count': '0'}, {'modelo': 'CNN_ResNet18', 'average_precision': '0.440797', 'roc_auc': '0.714658', 'balanced_accuracy': '0.653308', 'precision': '0.373840', 'recall': '0.597770', 'f1': '0.459724', 'fit_seconds': '1.197214', 'predict_seconds': '0.025149', 'ap_std': '0.034802', 'warnings_count': '0'}, {'modelo': 'ArbolDecision', 'average_precision': '0.419894', 'roc_auc': '0.720307', 'balanced_accuracy': '0.708044', 'precision': '0.428991', 'recall': '0.679554', 'f1': '0.525741', 'fit_seconds': '0.668875', 'predict_seconds': '0.016807', 'ap_std': '0.018127', 'warnings_count': '0'}, {'modelo': 'GaussianNB', 'average_precision': '0.299821', 'roc_auc': '0.647037', 'balanced_accuracy': '0.588311', 'precision': '0.264033', 'recall': '0.934572', 'f1': '0.411720', 'fit_seconds': '0.094426', 'predict_seconds': '0.016340', 'ap_std': '0.007276', 'warnings_count': '0'}]
display(pd.DataFrame(HISTORICO_TABLA))


,modelo,average_precision,roc_auc,balanced_accuracy,precision,recall,f1,fit_seconds,predict_seconds,ap_std,warnings_count
0,MLP,0.562697,0.810924,0.692717,0.574494,0.492937,0.528893,13.379517,0.011457,0.054918,0
1,SVM,0.560704,0.825072,0.676158,0.605437,0.434944,0.504704,7.583515,0.434863,0.047824,5
2,RegresionLogistica,0.547839,0.810890,0.741669,0.438855,0.769517,0.558860,0.750886,0.026128,0.053095,0
3,KNN,0.487864,0.785304,0.641199,0.587217,0.354647,0.441468,0.091354,0.147139,0.034729,0
4,ViT_B16,0.462416,0.738480,0.667968,0.399967,0.597026,0.478925,2.297461,0.027583,0.025392,0
5,CNN_ResNet18,0.440797,0.714658,0.653308,0.373840,0.597770,0.459724,1.197214,0.025149,0.034802,0
6,ArbolDecision,0.419894,0.720307,0.708044,0.428991,0.679554,0.525741,0.668875,0.016807,0.018127,0
7,GaussianNB,0.299821,0.647037,0.588311,0.264033,0.934572,0.411720,0.094426,0.016340,0.007276,0


## Entradas y partición
Descargar de la misma ejecución: checkpoint JSON y CSV de features y `split.csv` del modelo. Opcionalmente `provenance.json` para comprobar hashes. No sustituirlos por otro dataset del repositorio.

Checkpoint original: `gs://galmorphai-science/features/v1/d33b763ffed5bfa03e33b02a/ejecuciones/b73486a75d744e1eaf7c2ad1983fb2cf/checkpoint-08504.json`.
Resultados: `gs://galmorphai-science/models/v1/20261026T081634-6de4d5f5370a/`.

No hay acceso GCP automático ni escritura remota. El manifiesto evita que un cambio de orden/versión cambie inadvertidamente la partición.


In [12]:
# 1. Autenticar la sesión de Google Colab
from google.colab import auth
auth.authenticate_user()

# 2. Configurar el proyecto por defecto en gcloud
!gcloud config set project brave-server-511721-p7

# 3. Crear carpeta local
!mkdir -p datos

# 4. Descargar los archivos originales utilizando la herramienta oficial gcloud storage ya autenticada
!gcloud storage cp gs://galmorphai-science/features/v1/d33b763ffed5bfa03e33b02a/ejecuciones/b73486a75d744e1eaf7c2ad1983fb2cf/checkpoint-08504.json datos/
!gcloud storage cp gs://galmorphai-science/features/v1/d33b763ffed5bfa03e33b02a/ejecuciones/b73486a75d744e1eaf7c2ad1983fb2cf/checkpoint-08504.csv datos/
!gcloud storage cp gs://galmorphai-science/models/v1/20261026T081634-6de4d5f5370a/split.csv datos/

[environment: untagged] Read more to tag: g.co/cloud/project-env-tag.
Updated property [core/project].
Copying gs://galmorphai-science/features/v1/d33b763ffed5bfa03e33b02a/ejecuciones/b73486a75d744e1eaf7c2ad1983fb2cf/checkpoint-08504.json to file://datos/checkpoint-08504.json

Average throughput: 193.6MiB/s
Copying gs://galmorphai-science/features/v1/d33b763ffed5bfa03e33b02a/ejecuciones/b73486a75d744e1eaf7c2ad1983fb2cf/checkpoint-08504.csv to file://datos/checkpoint-08504.csv

Average throughput: 125.7MiB/s
Copying gs://galmorphai-science/models/v1/20261026T081634-6de4d5f5370a/split.csv to file://datos/split.csv


In [13]:
# Comprobar la descarga correcta
from pathlib import Path
print("Archivos descargados:")
for f in Path('datos').glob('*'):
    print(f"- {f.name} ({f.stat().st_size / 1024:.1f} KB)")

Archivos descargados:
- checkpoint-08504.json (59010.1 KB)
- split.csv (315.1 KB)
- checkpoint-08504.csv (20217.3 KB)


In [14]:
CHECKPOINT = Path('datos/checkpoint-08504.json')
SPLIT = Path('datos/split.csv')
PROVENANCE = Path('datos/provenance.json')  # opcional pero recomendado
OUT = Path('resultados_semana7') / (datetime.now(timezone.utc).strftime('%Y%m%dT%H%M%S')+'-'+uuid.uuid4().hex[:8])
N_ITER = 12
CFG = dict(MODEL_CFG)
for p in [CHECKPOINT, CHECKPOINT.with_suffix('.csv'), SPLIT]:
    if not p.exists(): raise FileNotFoundError(f'Falta entrada real: {p}. Asegúrese de ejecutar la celda de descarga anterior.')
datos, excluidos, cobertura, entrada = load_dataset(CHECKPOINT.with_suffix('.csv').read_bytes(), CHECKPOINT.read_bytes())
if PROVENANCE.exists():
    previo = json.loads(PROVENANCE.read_text())['entrada']
    for k in ['csv_sha256','json_sha256']:
        if previo[k] != entrada[k]: raise ValueError('Checkpoint distinto al de Semana 6: '+k)
split = pd.read_csv(SPLIT, dtype=str, keep_default_na=False)
if split.objeto_id.duplicated().any() or set(split.objeto_id)!=set(datos.objeto_id):
    raise ValueError('El manifiesto no coincide exactamente con objetos elegibles')
if set(split['split']) != {'train','test'}: raise ValueError('Particiones desconocidas')
datos = datos.merge(split[['objeto_id','split']], on='objeto_id', validate='one_to_one')
train = datos.loc[datos['split']=='train'].copy()
legacy_test = datos.loc[datos['split']=='test'].copy()
assert len(train)==5968 and len(legacy_test)==1492
X = train[PREDICTORS]; y = train.target_binario.to_numpy()
cv = list(StratifiedKFold(5, shuffle=True, random_state=42).split(X,y))
OUT.mkdir(parents=True, exist_ok=False)
split.to_csv(OUT/'split_historico.csv',index=False)
cobertura.to_csv(OUT/'cobertura.csv')
print('Entrenamiento:',len(train),'| Test histórico excluido de selección:',len(legacy_test))

Entrenamiento: 5968 | Test histórico excluido de selección: 1492


## Candidatos y optimización
Homogéneos: Random Forest, Extra Trees y AdaBoost con árboles. Heterogéneos: soft voting y stacking MLP+SVM, los dos mejores de Semana 6. Stacking lleva un pipeline completo **dentro de cada estimador** y produce predicciones internas fuera de fold; no usa `prefit`. El metamodelo es regresión logística. Se prueban ponderaciones de voting y regularización del metamodelo.

Se conserva MLP histórico y se añade una variante con 1500 iteraciones para revisar convergencia. Ninguna advertencia se oculta. Los parámetros heredados fueron seleccionados previamente sobre este entrenamiento: las métricas nuevas no son una estimación independiente. Para cuantificar generalización interna de todo el procedimiento se requiere CV anidada con selección de integrantes e hiperparámetros dentro de cada fold externo; no basta con reevaluar configuraciones ya elegidas.


In [15]:
from sklearn.base import clone
from sklearn.ensemble import RandomForestClassifier, ExtraTreesClassifier, AdaBoostClassifier, VotingClassifier, StackingClassifier

def tabular(estimator):
    p = make_model('ArbolDecision',CFG)
    p.set_params(model=estimator)
    return p

def candidatos():
    modelos = {name:make_model(name,CFG) for name in MODEL_NAMES}
    mlp = make_model('MLP',CFG).set_params(**HISTORICO['parametros_finales'])
    modelos['MLP_ajustado_S6'] = clone(mlp)
    modelos['MLP_1500'] = clone(mlp).set_params(model__max_iter=1500)
    bases = [('mlp',clone(modelos['MLP_1500'])),('svm',make_model('SVM',CFG))]
    modelos.update({
        'RandomForest':tabular(RandomForestClassifier(n_estimators=300,class_weight='balanced',random_state=42,n_jobs=2)),
        'ExtraTrees':tabular(ExtraTreesClassifier(n_estimators=300,class_weight='balanced',random_state=42,n_jobs=2)),
        'AdaBoost':tabular(AdaBoostClassifier(estimator=DecisionTreeClassifier(max_depth=1,random_state=42),random_state=42)),
        'Voting':VotingClassifier(estimators=bases,voting='soft',n_jobs=1),
        'Stacking':StackingClassifier(estimators=bases,final_estimator=LogisticRegression(max_iter=2000),
            cv=StratifiedKFold(5,shuffle=True,random_state=43),stack_method='predict_proba',n_jobs=1)
    })
    espacios = {
        'MLP_1500':{'model__alpha':[.0001,.001,.01], 'model__learning_rate_init':[.0001,.001]},
        'RandomForest':{'model__max_depth':[None,8,16], 'model__min_samples_leaf':[1,3,8], 'model__max_features':['sqrt',.5]},
        'ExtraTrees':{'model__max_depth':[None,8,16], 'model__min_samples_leaf':[1,3,8], 'model__max_features':['sqrt',.5]},
        'AdaBoost':{'model__n_estimators':[50,100,200], 'model__learning_rate':[.03,.1,1.], 'model__estimator__max_depth':[1,2]},
        'Voting':{'weights':[(1,1),(2,1),(1,2)]},
        'Stacking':{'final_estimator__C':[.01,.1,1,10,100]}
    }
    return modelos, espacios

def evaluar_cv(estimator,X,y,folds):
    oof=np.full(len(y),np.nan); rows=[]
    for fold,(a,b) in enumerate(folds):
        model=clone(estimator)
        with warnings.catch_warnings(record=True) as caught:
            warnings.simplefilter('always')
            t=perf_counter(); model.fit(X.iloc[a],y[a]); fit=perf_counter()-t
            t=perf_counter(); p=model.predict_proba(X.iloc[b])[:,1]; infer=perf_counter()-t
        oof[b]=p
        m=metrics(y[b],p,.5); m.pop('confusion_matrix')
        rows.append(dict(m,fold=fold,fit_seconds=fit,predict_seconds=infer,
                         warnings=' | '.join(sorted(set(str(w.message) for w in caught)))))
    assert np.isfinite(oof).all()
    table=pd.DataFrame(rows)
    stats=table.select_dtypes('number').drop(columns='fold').mean().to_dict()
    stats.update(ap_std=float(table.average_precision.std()),warnings_count=int(table.warnings.ne('').sum()))
    return stats,oof,table


In [16]:
modelos, espacios = candidatos()
comparacion=[]; detalles=[]; busqueda=[]; elegidos={}; oofs={}
for name,base in modelos.items():
    candidates=[{}]
    if name in espacios:
        total=int(np.prod([len(v) for v in espacios[name].values()]))
        candidates+=list(ParameterSampler(espacios[name],n_iter=min(N_ITER,total),random_state=42))
    start=perf_counter(); best=None
    for i,params in enumerate(candidates):
        est=clone(base).set_params(**params)
        stats,oof,detail=evaluar_cv(est,X,y,cv)
        busqueda.append(dict(modelo=name,candidato=i,parametros=json.dumps(params),**stats))
        detalles.append(detail.assign(modelo=name,candidato=i))
        if best is None or stats['average_precision']>best[0]['average_precision']:
            best=(stats,oof,est,params)
    stats,oof,est,params=best
    comparacion.append(dict(modelo=name,**stats,busqueda_wall_seconds=perf_counter()-start,parametros=json.dumps(params)))
    elegidos[name]=est; oofs[name]=oof
    pd.DataFrame(busqueda).to_csv(OUT/'busqueda.csv',index=False)
    print(name, 'AP CV:',round(stats['average_precision'],4),'folds con alertas:',stats['warnings_count'])
tabla=pd.DataFrame(comparacion).sort_values(['average_precision','predict_seconds','modelo'],ascending=[False,True,True])
tabla.to_csv(OUT/'comparacion_semana7.csv',index=False)
pd.concat(detalles).to_csv(OUT/'folds_y_advertencias.csv',index=False)
historica=pd.DataFrame(HISTORICO_TABLA).assign(origen='Semana6 histórico; CNN/ViT tiempo de cabeza')
actual=tabla.assign(origen='Semana7 CV selección; tiempo medido en entorno actual')
unificada=pd.concat([historica,actual],ignore_index=True)
unificada['average_precision']=pd.to_numeric(unificada['average_precision'])
unificada.sort_values('average_precision',ascending=False).to_csv(OUT/'comparacion_completa.csv',index=False)
display(tabla)
print('Comparación exploratoria; no demuestra mejora significativa. Revisar folds_y_advertencias.csv.')

RegresionLogistica AP CV: 0.5548 folds con alertas: 0
SVM AP CV: 0.5619 folds con alertas: 5
ArbolDecision AP CV: 0.4225 folds con alertas: 1
KNN AP CV: 0.4863 folds con alertas: 0
GaussianNB AP CV: 0.3071 folds con alertas: 0
MLP AP CV: 0.5601 folds con alertas: 1
MLP_ajustado_S6 AP CV: 0.5846 folds con alertas: 5
MLP_1500 AP CV: 0.5905 folds con alertas: 0
RandomForest AP CV: 0.5638 folds con alertas: 0
ExtraTrees AP CV: 0.5725 folds con alertas: 0
AdaBoost AP CV: 0.5379 folds con alertas: 0
Voting AP CV: 0.5993 folds con alertas: 5
Stacking AP CV: 0.5976 folds con alertas: 5


,modelo,average_precision,roc_auc,balanced_accuracy,precision,recall,f1,prevalencia_positiva,fit_seconds,predict_seconds,ap_std,warnings_count,busqueda_wall_seconds,parametros
11,Voting,0.599323,0.835898,0.690756,0.634847,0.458736,0.531697,0.225369,30.442395,0.335111,0.031140,5,638.222925,"{""weights"": [2, 1]}"
12,Stacking,0.597626,0.836526,0.518715,0.811255,0.040892,0.077499,0.225369,148.959089,0.324192,0.028524,5,4445.984146,"{""final_estimator__C"": 0.01}"
7,MLP_1500,0.590494,0.831366,0.691350,0.625507,0.464684,0.531714,0.225369,26.722029,0.005142,0.036392,0,634.347110,"{""model__learning_rate_init"": 0.0001, ""model__..."
6,MLP_ajustado_S6,0.584621,0.832530,0.678144,0.629014,0.430483,0.509974,0.225369,11.712970,0.008241,0.028425,5,58.712644,{}
9,ExtraTrees,0.572460,0.826267,0.737813,0.506233,0.664684,0.574356,0.225369,3.720196,0.079485,0.014609,0,198.612814,"{""model__min_samples_leaf"": 8, ""model__max_fea..."
8,RandomForest,0.563780,0.825203,0.744503,0.500281,0.689963,0.579725,0.225369,4.287122,0.088714,0.015412,0,825.364449,"{""model__min_samples_leaf"": 8, ""model__max_fea..."
1,SVM,0.561853,0.827783,0.677786,0.603650,0.440149,0.508010,0.225369,4.390351,0.244398,0.020605,5,23.241013,{}
5,MLP,0.560076,0.815274,0.689588,0.592342,0.474349,0.526199,0.225369,7.278967,0.005330,0.040403,1,36.497080,{}
0,RegresionLogistica,0.554842,0.815145,0.746422,0.444319,0.776208,0.564831,0.225369,0.622852,0.012859,0.024843,0,3.367362,{}
10,AdaBoost,0.537915,0.812180,0.669621,0.596581,0.422305,0.494322,0.225369,9.257413,0.053321,0.017860,0,514.750843,"{""model__n_estimators"": 200, ""model__learning_..."


Comparación exploratoria; no demuestra mejora significativa. Revisar folds_y_advertencias.csv.


## Fijar modelo y umbral antes de datos nuevos
La regla automática prioriza AP, con desempate exacto por inferencia y nombre. Revisar convergencia y necesidades de seguimiento astronómico antes de declarar el modelo final. F1 OOF maximizado es un valor de ajuste, no una evaluación independiente. No se exige que un ensamble gane: puede conservarse el individual.


In [17]:
winner=tabla.iloc[0].modelo
threshold,f1_ajuste=select_threshold(y,oofs[winner])
selected=clone(elegidos[winner])
t=perf_counter(); selected.fit(X,y); final_fit=perf_counter()-t
metadata=dict(modelo=winner,umbral=threshold,f1_oof_ajuste=f1_ajuste,final_fit_seconds=final_fit,
    entrada=entrada,features=PREDICTORS,labels=LABEL_MAP,test_historico_inspeccionado=True,
    versiones={p:version(p) for p in ['numpy','pandas','scikit-learn','joblib']},
    parametros=tabla.iloc[0].parametros,alcance='Presencia de anillos en población con QC aprobado')
(OUT/'seleccion.json').write_text(json.dumps(metadata,ensure_ascii=False,indent=2))
joblib.dump(dict(pipeline=selected,threshold=threshold,features=PREDICTORS,metadata=metadata),OUT/'modelo.joblib')
pd.DataFrame({'objeto_id':train.objeto_id.to_numpy(),'target_binario':y,**oofs}).to_csv(OUT/'oof.csv',index=False)
print('Modelo candidato fijado:',winner,'umbral:',threshold)


/usr/local/lib/python3.13/dist-packages/sklearn/svm/_base.py:236: FutureWarning: The `probability` parameter was deprecated in 1.9 and will be removed in version 1.11. Use `CalibratedClassifierCV(SVC(), ensemble=False)` instead of `SVC(probability=True)`
  warnings.warn(


Modelo candidato fijado: Voting umbral: 0.31999999999999995


## Evaluación final externa (pendiente de datos)
Proporcionar un checkpoint CSV/JSON con el mismo esquema, extraído con la misma receta y con galaxias nunca usadas en desarrollo. El código comprueba IDs, pero la independencia espacial, imágenes duplicadas con IDs distintos y ausencia de inspección previa requieren verificar procedencia. No crear un “test nuevo” cambiando la semilla sobre las mismas 7460 galaxias.

Ejecutar una vez tras fijar selección. No volver a ajustar con sus resultados. Se generan métricas, curvas, matriz, importancia por permutación y sensibilidad por subtipo; no es clasificación multiclase. La importancia puede ser inestable con predictores correlacionados y no es causal.


## Entrega y límites
Entregar comparación ordenada por AP, búsquedas, tiempos medios por fold y coste total de búsqueda separados, alertas, configuración, partición, cobertura y modelo. Los resultados históricos no se mezclan con test externo ni con métricas al umbral optimizado.

El objetivo 3.5 se acredita sólo si la evidencia muestra mejora; no prometerla. Para un contraste pareado independiente, fijar previamente un baseline MLP, entrenarlo sin datos externos y evaluar ambos una sola vez sobre el mismo conjunto externo, con intervalo bootstrap de la diferencia de AP por objeto o grupo espacial. Esta extensión estadística no está implementada aquí. El objetivo 3.6 queda pendiente hasta incorporar datos nuevos; CV anidada sería una alternativa interna con sus límites por decisiones históricas.

Apoyo de IA: extracción de evidencia, auditoría y preparación de este notebook. El equipo debe verificar procedencia externa, convergencia, interpretación y ejecución real.

Referencias: [StackingClassifier](https://scikit-learn.org/stable/modules/generated/sklearn.ensemble.StackingClassifier.html), [validación anidada](https://scikit-learn.org/stable/auto_examples/model_selection/plot_nested_cross_validation_iris.html).


## Declaración de uso de inteligencia artificial

### Herramienta y alcance del apoyo

Se utilizó **ChatGPT / Codex, de OpenAI**, como herramienta de apoyo para la revisión metodológica, la preparación de código y la documentación del proyecto GalMorphAI, Semana 7 — Avance 5. La asistencia de IA no sustituye la revisión académica ni la responsabilidad del equipo sobre la entrega.

| Actividad | Apoyo proporcionado por la IA | Alcance en el notebook |
|---|---|---|
| Revisión de Semana 6 | Extracción de modelos, métricas, tiempos, etiquetas y particiones a partir del notebook proporcionado. | Resultados históricos y definición del problema binario. |
| Diseño de ensambles | Propuesta de Random Forest, Extra Trees, AdaBoost, voting y stacking con MLP y SVM. | Candidatos y optimización. |
| Implementación | Apoyo en la generación y adaptación de pipelines, búsquedas de hiperparámetros, validación cruzada, registro de tiempos y exportación de resultados. | Código de comparación y selección. |
| Evaluación metodológica | Identificación de riesgos de fuga, reutilización de validación y limitaciones del test previamente inspeccionado. | Particiones, selección de umbral y límites de evaluación. |
| Documentación | Apoyo en la explicación de métricas, requerimientos de ejecución y elaboración de esta declaración. | Celdas explicativas y documentación de la entrega. |

### Verificación y responsabilidad

- En la preparación de la versión base se realizaron pruebas sintéticas de los candidatos, incluidos voting y stacking. Estas pruebas verifican funcionamiento básico; no equivalen a validar el desempeño con datos reales ni certifican modificaciones posteriores.
- Los resultados históricos proceden de las salidas del notebook de Semana 6 proporcionado. Las métricas nuevas deben proceder de la ejecución del código sobre los datos del proyecto; no deben reemplazarse por estimaciones o valores generados por IA.
- Corresponde al equipo comprobar las entradas, las etiquetas, las particiones, las advertencias de entrenamiento, los resultados y su interpretación antes de la entrega. Esta declaración no afirma que todas esas comprobaciones ya se hayan completado.
- La elección del modelo final y la justificación científica y académica son responsabilidad del equipo.

### Decisiones metodológicas apoyadas por IA

1. Mantener Average Precision (AP) como métrica principal para conservar coherencia con Semana 6; utilizar F1 para el ajuste del umbral y otras métricas como complemento.
2. Incorporar ensambles homogéneos y heterogéneos, utilizando MLP y SVM como integrantes de los ensambles heterogéneos por su rendimiento previo.
3. Mantener el preprocesamiento dentro de los pipelines de validación y utilizar predicciones fuera de fold para el metamodelo de stacking.
4. Reconocer que el test histórico fue inspeccionado previamente y que no se dispone de datos externos nuevos. No presentar los resultados históricos como una evaluación externa independiente ni garantizar una mejora del ensamble.

### Referencia de la herramienta

OpenAI. (2026). *ChatGPT / Codex* [Asistente de inteligencia artificial]. https://chatgpt.com/
